# Phase 1 — Embedding Extraction with BiomedCLIP

This notebook runs **phase 1** of the SAE pipeline:
it extracts visual and text embeddings from the IU X-Ray dataset
using the `chuhac/BiomedCLIP-vit-bert-hf` model.

**Input:**
- `data/iu_xray/images/images_normalized/` — 7 470 PNG chest X-rays
- `data/iu_xray/reports/indiana_reports.csv` — clinical reports

**Output:**
- `embeddings/visual_embeddings.pt` — tensor `(N_img * (1 + N_aug), 512)`
- `embeddings/text_embeddings.pt`  — tensor `(N_rep, 512)`

**Device:** MPS (Apple Silicon) → CUDA → CPU fallback

## 0. Setup & Configuration

In [1]:
import sys
from pathlib import Path

import torch

# Traverse up to the project root
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))  # so the `xai_datasets` package is importable

# Select compute device (MPS > CUDA > CPU)
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

print(f'Project root : {PROJECT_ROOT}')
print(f'PyTorch      : {torch.__version__}')
print(f'Device       : {DEVICE}')

Project root : /teamspace/studios/this_studio/xai-project-5
PyTorch      : 2.12.0+cu126
Device       : cuda


In [2]:
import config

# Local IU X-Ray layout — matches the default EmbeddingConfig and the staging
# produced by xai_datasets/download_iu_xray.py on this machine (images/ and
# reports/ are siblings under data/iu_xray/, not nested under the Kaggle zip).
DATASET_ROOT = PROJECT_ROOT / 'data' / 'iu_xray'

# Model / runtime config. VLMConfig no longer carries I/O paths — those live in
# EmbeddingConfig since the VLMConfig/EmbeddingConfig split.
vlm_cfg = config.VLMConfig(
    device=DEVICE,
    batch_size=32,   # keep <64 for MPS stability
    num_workers=0,   # 0 = main process (required on macOS)
)

# I/O config: where to read images/reports and where to write embeddings.
emb_cfg = config.EmbeddingConfig(
    image_dir=str(DATASET_ROOT / 'images' / 'images_normalized'),
    reports_dir=str(DATASET_ROOT / 'reports'),  # indiana_reports.csv lives here
    output_base=str(PROJECT_ROOT / 'embeddings'),
)

print('=== Configs ===')
print(f'  model       : {vlm_cfg.model_name}')
print(f'  device      : {vlm_cfg.device}')
print(f'  batch_size  : {vlm_cfg.batch_size}')
print(f'  image_dir   : {emb_cfg.image_dir}')
print(f'  reports_dir : {emb_cfg.reports_dir}')
print(f'  visual_out  : {emb_cfg.visual_output_path}')
print(f'  text_out    : {emb_cfg.text_output_path}')

=== Configs ===
  model       : chuhac/BiomedCLIP-vit-bert-hf
  device      : cuda
  batch_size  : 32
  image_dir   : /teamspace/studios/this_studio/xai-project-5/data/iu_xray/images/images_normalized
  reports_dir : /teamspace/studios/this_studio/xai-project-5/data/iu_xray/reports
  visual_out  : /teamspace/studios/this_studio/xai-project-5/embeddings/standard/visual_embeddings.pt
  text_out    : /teamspace/studios/this_studio/xai-project-5/embeddings/standard/text_embeddings.pt


In [3]:
images_dir = DATASET_ROOT / 'images' / 'images_normalized'
reports_csv = DATASET_ROOT / 'reports' / 'indiana_reports.csv'

print('=== IU X-Ray staging ===')
print(f'Images directory : {images_dir}')
print(f'Reports CSV      : {reports_csv}')

if not images_dir.exists():
    raise FileNotFoundError(f'Missing images directory: {images_dir}')

if not reports_csv.exists():
    raise FileNotFoundError(f'Missing report CSV: {reports_csv}')

n_images = len(list(images_dir.glob('*.png')))
print(f'PNG files found  : {n_images}')
print('Dataset staged correctly.')

=== IU X-Ray staging ===
Images directory : /teamspace/studios/this_studio/xai-project-5/data/iu_xray/images/images_normalized
Reports CSV      : /teamspace/studios/this_studio/xai-project-5/data/iu_xray/reports/indiana_reports.csv
PNG files found  : 7470
Dataset staged correctly.


## 1. Data Verification

In [ ]:
from xai_datasets.spec import get_dataset

# Resolve the active dataset from the spec registry (config.active_dataset.name,
# default "iu_xray"). Phase 0: only IU X-Ray is registered; PadChest/ROCOv2 land
# in later phases. The spec carries the Dataset classes + input paths, so this
# cell is dataset-agnostic.
spec = get_dataset(config.active_dataset.name)
print(f'Active dataset: {spec.name} ({spec.language}, {spec.domain})')

image_dataset = spec.image_dataset_cls(spec.image_dir)
text_dataset  = spec.text_dataset_cls(spec.text_source)

print(f'Images found  : {len(image_dataset)}')
print(f'Reports found : {len(text_dataset)}')

# Visual sample
img, path = image_dataset[0]
print(f'\nSample image: {Path(path).name} — mode={img.mode}, size={img.size}')

# Text sample
text, uid = text_dataset[0]
print(f'Sample text : uid={uid}')
print(f'  → {text[:120]}...')

## 2. Load BiomedCLIP

In [5]:
from transformers import AutoModel, AutoProcessor
from transformers.models.clip.configuration_clip import CLIPConfig
import transformers.models.clip.modeling_clip as _clip_mod

# ── Compatibility patches for BiomedCLIP custom code + newer transformers ──

# 1) CLIPConfig now rejects positional args; the cached BiomedCLIP config passes them.
_orig_clip_init = CLIPConfig.__init__
def _clip_init_compat(self, *args, **kwargs):
    if args:
        names = ['text_config', 'vision_config', 'projection_dim', 'logit_scale_init_value']
        for name, val in zip(names, args):
            if name not in kwargs:
                kwargs[name] = val
    _orig_clip_init(self, **kwargs)
CLIPConfig.__init__ = _clip_init_compat

# 2) BiomedCLIP uses `from modeling_clip import *` but __all__ restricts exports.
if hasattr(_clip_mod, '__all__'):
    del _clip_mod.__all__

print(f'Loading model: {vlm_cfg.model_name} ...')
processor = AutoProcessor.from_pretrained(vlm_cfg.processor_name, trust_remote_code=True)
model     = AutoModel.from_pretrained(vlm_cfg.model_name, trust_remote_code=True)

# 3) BiomedCLIP text tower checks config.is_decoder (BERT-style) but CLIPTextConfig lacks it.
if not hasattr(model.text_model.config, 'is_decoder'):
    model.text_model.config.is_decoder = False

# 4) Fix corrupted position_ids buffer (PyTorch 2.12 + persistent=False bug).
import torch
max_pos = model.text_model.config.max_position_embeddings
model.text_model.embeddings.position_ids = torch.arange(max_pos).unsqueeze(0)

model = model.to(DEVICE)
model.eval()

print(f'Model on {DEVICE} — total parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading model: chuhac/BiomedCLIP-vit-bert-hf ...
Model on cuda — total parameters: 195,902,721


## 3. Visual Embedding Extraction

*Note: If `config.augmentation.enabled` is True, the dataset is automatically wrapped in `AugmentedImageDataset` to generate augmented samples on-the-fly, expanding the output tensor without generating temporary image files.*

In [6]:
import json

from embedding_extraction.extract_embeddings import extract_visual_embeddings

extract_visual_embeddings(model, processor, image_dataset, vlm_cfg, emb_cfg)

# Verify output
visual_emb = torch.load(emb_cfg.visual_output_path, map_location='cpu', weights_only=True)
print(f'\nvisual_embeddings.pt : {visual_emb.shape}  dtype={visual_emb.dtype}')
print(f'  mean norm          : {visual_emb.norm(dim=-1).mean():.4f}  (expected ~1.0)')

# Extraction also writes a visual_image_ids.json sidecar (basenames aligned
# row-for-row), consumed downstream by the train/test split + generate_explanations.
ids_path = emb_cfg.visual_output_path.with_name('visual_image_ids.json')
print(f'  image-id sidecar   : {ids_path.name}  ({len(json.loads(ids_path.read_text()))} ids)')


Found 7470 images. Starting extraction...


Images Processing:   0%|          | 0/234 [00:00<?, ?it/s]

Images Processing: 100%|██████████| 234/234 [17:32<00:00,  4.50s/it]

Images Embedding Extraction completed. Saving on /teamspace/studios/this_studio/xai-project-5/embeddings/standard/visual_embeddings.pt.

visual_embeddings.pt : torch.Size([7470, 512])  dtype=torch.float32
  mean norm          : 1.0000  (expected ~1.0)
  image-id sidecar   : visual_image_ids.json  (7470 ids)


## 4. Text Embedding Extraction

In [7]:
from embedding_extraction.extract_embeddings import extract_text_embeddings

extract_text_embeddings(model, processor, text_dataset, vlm_cfg, emb_cfg)

# Verify output
text_emb = torch.load(emb_cfg.text_output_path, map_location='cpu', weights_only=True)
print(f'\ntext_embeddings.pt : {text_emb.shape}  dtype={text_emb.dtype}')
print(f'  mean norm        : {text_emb.norm(dim=-1).mean():.4f}  (expected ~1.0)')


Found 3851 reports. Starting extraction...


Reports Processing: 100%|██████████| 121/121 [00:28<00:00,  4.29it/s]

Reports Embedding Extraction completed. Saving on /teamspace/studios/this_studio/xai-project-5/embeddings/standard/text_embeddings.pt.

text_embeddings.pt : torch.Size([3851, 512])  dtype=torch.float32
  mean norm        : 1.0000  (expected ~1.0)


## 5. Embedding Quality Check

In [8]:
import numpy as np

# Load from disk
v_emb = torch.load(emb_cfg.visual_output_path, map_location='cpu', weights_only=True)
t_emb = torch.load(emb_cfg.text_output_path, map_location='cpu', weights_only=True)

print('═══ Embedding Statistics ═══\n')
print(f'Visual embeddings : {v_emb.shape}  ({v_emb.dtype})')
print(f'Text embeddings   : {t_emb.shape}  ({t_emb.dtype})')

# Norms (should be ~1.0 for L2-normalized)
v_norms = v_emb.norm(dim=-1)
t_norms = t_emb.norm(dim=-1)
print(f'\n── Norms ──')
print(f'Visual: mean={v_norms.mean():.6f}, std={v_norms.std():.6f}, min={v_norms.min():.6f}, max={v_norms.max():.6f}')
print(f'Text  : mean={t_norms.mean():.6f}, std={t_norms.std():.6f}, min={t_norms.min():.6f}, max={t_norms.max():.6f}')

# Distribution of values
print(f'\n── Value Distribution ──')
print(f'Visual: mean={v_emb.mean():.6f}, std={v_emb.std():.6f}, min={v_emb.min():.6f}, max={v_emb.max():.6f}')
print(f'Text  : mean={t_emb.mean():.6f}, std={t_emb.std():.6f}, min={t_emb.min():.6f}, max={t_emb.max():.6f}')

# Sparsity (fraction of near-zero activations)
v_sparse = (v_emb.abs() < 0.01).float().mean()
t_sparse = (t_emb.abs() < 0.01).float().mean()
print(f'\n── Sparsity (|x| < 0.01) ──')
print(f'Visual: {v_sparse:.4f} ({v_sparse*100:.1f}%)')
print(f'Text  : {t_sparse:.4f} ({t_sparse*100:.1f}%)')

# Cosine similarity within each modality (sample)
def avg_pairwise_cosine(emb, n_sample=500):
    idx = torch.randperm(len(emb))[:n_sample]
    sample = emb[idx]
    sim = sample @ sample.T
    mask = ~torch.eye(n_sample, dtype=bool)
    return sim[mask].mean().item(), sim[mask].std().item()

v_cos_mean, v_cos_std = avg_pairwise_cosine(v_emb)
t_cos_mean, t_cos_std = avg_pairwise_cosine(t_emb)
print(f'\n── Avg Pairwise Cosine Similarity (sample 500) ──')
print(f'Visual-Visual: {v_cos_mean:.4f} ± {v_cos_std:.4f}')
print(f'Text-Text    : {t_cos_mean:.4f} ± {t_cos_std:.4f}')

# Cross-modal similarity
cross_sim = (v_emb[:500] @ t_emb[:500].T).mean().item()
print(f'Cross-modal  : {cross_sim:.4f}')

# File sizes
import os
v_size = os.path.getsize(emb_cfg.visual_output_path) / (1024**2)
t_size = os.path.getsize(emb_cfg.text_output_path) / (1024**2)
print(f'\n── File Sizes ──')
print(f'visual_embeddings.pt : {v_size:.2f} MB')
print(f'text_embeddings.pt   : {t_size:.2f} MB')
print(f'Total                : {v_size + t_size:.2f} MB')

═══ Embedding Statistics ═══

Visual embeddings : torch.Size([7470, 512])  (torch.float32)
Text embeddings   : torch.Size([3851, 512])  (torch.float32)

── Norms ──
Visual: mean=1.000000, std=0.000000, min=1.000000, max=1.000000
Text  : mean=1.000000, std=0.000000, min=1.000000, max=1.000000

── Value Distribution ──
Visual: mean=-0.000478, std=0.044192, min=-0.341880, max=0.365170
Text  : mean=0.000665, std=0.044189, min=-0.235362, max=0.550903

── Sparsity (|x| < 0.01) ──
Visual: 0.2136 (21.4%)
Text  : 0.2379 (23.8%)

── Avg Pairwise Cosine Similarity (sample 500) ──
Visual-Visual: 0.7954 ± 0.1133
Text-Text    : 0.7933 ± 0.0815
Cross-modal  : 0.3532

── File Sizes ──
visual_embeddings.pt : 14.59 MB
text_embeddings.pt   : 7.52 MB
Total                : 22.11 MB
